# BigQuery: Conversational Analytics with TimesFM (March 2026 Suite)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_conversational_analytics_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_conversational_analytics_demo.ipynb)

**What you'll see:** An AI agent that talks to BigQuery — catching anomalies, investigating receipts, and forecasting sales, all through natural language.

## Scenario: From Crisis to Clarity
A data analyst at a retail company notices something is off. Instead of writing SQL manually, they ask a **conversational agent** to investigate:

| Step | Feature | Narrative |
|------|---------|-----------|
| 1 | `AI.DETECT_ANOMALIES` | **"Something is wrong."** — Flag today's $20K spike against a $200/day baseline |
| 2 | `ObjectRef` | **"Let me check the receipt."** — Pull the receipt image from Cloud Storage |
| 3 | `AI.FORECAST` | **"What happens next?"** — Predict the next 7 days assuming the anomaly was a one-off |

### Key Technologies
- **BigQuery AI Functions** — [`AI.FORECAST`](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-forecast) and [`AI.DETECT_ANOMALIES`](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-detect-anomalies) powered by [TimesFM](https://research.google/blog/a-decoder-only-foundation-model-for-time-series-forecasting/) — zero-shot, no model training required
- **ObjectRef** — Reference Cloud Storage files (images, PDFs) directly from BigQuery SQL
- **ADK v1.28.0** — [BigQueryToolset](https://github.com/google/adk-python/releases/tag/v1.28.0) with job labels for cost attribution
- **Gemini 3.1 Pro** (Preview) — Powering the conversational agent

### Requirements
- `google-adk >= 1.28.0` and `google-genai >= 1.69.0`
- BigQuery, Cloud Storage, and Vertex AI APIs enabled
- A Google Cloud project with billing enabled

In [ ]:
%pip install "google-adk>=1.28.0" google-genai google-cloud-bigquery google-cloud-storage nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'US' # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. Enable APIs

In [ ]:
!gcloud services enable bigquery.googleapis.com storage.googleapis.com aiplatform.googleapis.com --project={project_id}
!gcloud config set project {project_id}
print("APIs enabled and project configured.")

### 3. Create Demo Data

This cell creates:
- A **partitioned BigQuery table** with 365 days of simulated retail sales (~$200/day with natural variance)
- A **$20,000 anomaly** injected on today's date (the agent should catch this!)
- A **GCS bucket** with a dummy receipt image for the ObjectRef demo

> **Why 365 days?** `AI.DETECT_ANOMALIES` uses Google's [TimesFM](https://research.google/blog/a-decoder-only-foundation-model-for-time-series-forecasting/) foundation model, which needs 128+ historical data points for accurate anomaly detection.

In [ ]:
from google.cloud import bigquery, storage
from datetime import datetime, timedelta
import random

def setup_infrastructure():
    bq_client = bigquery.Client(project=project_id, location=location)
    storage_client = storage.Client(project=project_id)

    dataset_id = f"{project_id}.march_demo"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = location
    bq_client.create_dataset(dataset, exists_ok=True)
    print(f"[1/4] Dataset '{dataset_id}' ready.")

    table_id = f"{dataset_id}.sales_data"
    schema = [
        bigquery.SchemaField("sale_date", "DATE"),
        bigquery.SchemaField("product_id", "STRING"),
        bigquery.SchemaField("amount", "FLOAT"),
        bigquery.SchemaField("receipt_id", "STRING"),
    ]
    random.seed(42)
    data = [
        {"sale_date": (datetime.now() - timedelta(days=i)).date().isoformat(),
         "product_id": f"PROD_{i%5}",
         "amount": round(200.0 + random.uniform(-50, 50), 2),
         "receipt_id": f"R_{1000+i}"}
        for i in range(365)
    ]
    data.append({"sale_date": datetime.now().date().isoformat(),
                 "product_id": "PROD_999", "amount": 20000.0, "receipt_id": "R_ANOMALY"})
    print(f"[2/4] Generated {len(data)} rows (365-day baseline + anomaly).")

    job_config = bigquery.LoadJobConfig(
        schema=schema, write_disposition="WRITE_TRUNCATE",
        time_partitioning=bigquery.TimePartitioning(field="sale_date"),
    )
    load_job = bq_client.load_table_from_json(data, table_id, job_config=job_config)
    load_job.result()
    print(f"[3/4] Table '{table_id}' loaded: {load_job.output_rows} rows.")

    bucket_name = f"{project_id}-receipts"
    bucket = (storage_client.create_bucket(bucket_name, location=location)
              if not storage_client.lookup_bucket(bucket_name)
              else storage_client.get_bucket(bucket_name))
    bucket.blob("receipts/R_ANOMALY.jpg").upload_from_string(
        b"Dummy Receipt Image Data", content_type="image/jpeg")
    print(f"[4/4] Receipt uploaded to gs://{bucket_name}/receipts/R_ANOMALY.jpg")
    print(f"\nReady! Table and GCS bucket are set up.")

setup_infrastructure()

### 4. Initialize the Conversational Agent

The agent combines two types of tools:
- **BigQuery Toolset** (ADK built-in) — for general schema exploration and ad-hoc SQL
- **Custom BQML tools** — deterministic SQL wrappers around `AI.FORECAST`, `AI.DETECT_ANOMALIES`, and receipt lookup

This hybrid pattern ensures the AI functions execute reliably while still giving the agent flexibility for exploratory queries.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.adk.tools.bigquery import BigQueryToolset
from google.adk.tools.bigquery.config import BigQueryToolConfig
from google.cloud import bigquery
from google.genai import types
from datetime import date, datetime

bq_toolset = BigQueryToolset(bigquery_tool_config=BigQueryToolConfig(
    job_labels={"ca-bq-job": "true", "demo": "march-2026-suite"}
))

os.environ["GOOGLE_CLOUD_LOCATION"] = "global"  # Gemini 3.1 Pro Preview endpoint

fq_table = f"{project_id}.march_demo.sales_data"
gcs_bucket = f"gs://{project_id}-receipts"
bq_client = bigquery.Client(project=project_id)

def _make_serializable(obj):
    if isinstance(obj, (date, datetime)):
        return obj.isoformat()
    return obj

def _run_query(sql: str, label: str, job_config=None) -> list[dict]:
    print(f"[SQL — {label}]:\n{sql.strip()}\n")
    rows = list(bq_client.query(sql, job_config=job_config).result())
    return [{k: _make_serializable(v) for k, v in dict(row).items()} for row in rows]

def demo_forecast_sales(horizon: int = 7) -> dict:
    """Forecast total daily sales for the next N days using BigQuery AI.FORECAST.
    Args:
        horizon: Number of days to forecast (default 7).
    """
    sql = f"""
    SELECT * FROM AI.FORECAST(
      (SELECT sale_date, SUM(amount) AS total_sales
       FROM `{fq_table}` GROUP BY sale_date),
      timestamp_col => 'sale_date',
      data_col => 'total_sales',
      horizon => {int(horizon)},
      confidence_level => 0.95
    )"""
    return {"forecast": _run_query(sql, "demo_forecast_sales")}

def demo_detect_anomalies(anomaly_prob_threshold: float = 0.8) -> dict:
    """Detect anomalies in daily sales using BigQuery AI.DETECT_ANOMALIES (TimesFM).
    Args:
        anomaly_prob_threshold: Anomaly probability threshold (default 0.8).
    """
    sql = f"""
    WITH daily_sales AS (
      SELECT sale_date AS date, CAST(SUM(amount) AS FLOAT64) AS total_sales
      FROM `{fq_table}` GROUP BY 1
    )
    SELECT * FROM AI.DETECT_ANOMALIES(
      (SELECT * FROM daily_sales WHERE date < DATE_SUB(CURRENT_DATE(), INTERVAL 14 DAY) ORDER BY date),
      (SELECT * FROM daily_sales WHERE date >= DATE_SUB(CURRENT_DATE(), INTERVAL 14 DAY) ORDER BY date),
      data_col => 'total_sales',
      timestamp_col => 'date',
      anomaly_prob_threshold => {float(anomaly_prob_threshold)}
    )
    ORDER BY time_series_timestamp DESC"""
    return {"anomalies": _run_query(sql, "demo_detect_anomalies")}

def demo_lookup_receipt(receipt_id: str) -> dict:
    """Look up a sale by receipt_id and return the GCS receipt image link.
    Args:
        receipt_id: The receipt ID to look up (e.g. 'R_ANOMALY').
    """
    sql = f"""
    SELECT sale_date, product_id, amount, receipt_id,
           CONCAT(@gcs_bucket, '/receipts/', receipt_id, '.jpg') AS receipt_gcs_uri
    FROM `{fq_table}`
    WHERE receipt_id = @receipt_id"""
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("receipt_id", "STRING", receipt_id),
        bigquery.ScalarQueryParameter("gcs_bucket", "STRING", gcs_bucket),
    ])
    return {"results": _run_query(sql, "demo_lookup_receipt", job_config=job_config)}

agent = Agent(
    model="gemini-3.1-pro-preview",
    name="ConversationalAnalyst",
    instruction=f"""You are a senior BigQuery analyst for project '{project_id}'.
    Table: `{fq_table}` (partitioned on sale_date, 365 days of daily retail sales).
    Use your specialized tools: demo_forecast_sales, demo_detect_anomalies, demo_lookup_receipt.
    For general BQ queries use the BigQuery toolset. Never ask for project ID or table names.
    Present results in clear table format.""",
    tools=[bq_toolset, demo_forecast_sales, demo_detect_anomalies, demo_lookup_receipt]
)

runner = Runner(
    agent=agent,
    session_service=InMemorySessionService(),
    app_name="conversational_analytics_demo",
    auto_create_session=True
)

async def run_agent(prompt: str):
    message = types.Content(parts=[types.Part(text=prompt)], role='user')
    async for event in runner.run_async(
        user_id="partner_user", session_id="march_session", new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent: {part.text}")
                if part.function_call:
                    print(f"  >> Tool: '{part.function_call.name}' | Args: {part.function_call.args}")

print("Agent ready.")

### 5a. "Something is wrong" — Detect Anomalies

The agent uses `AI.DETECT_ANOMALIES` (TimesFM) to evaluate the last 14 days against the historical baseline. It should flag today's **$20,000 spike** with probability 1.0.

> **Under the hood:** TimesFM builds a zero-shot forecast from 351 days of history, then compares actual values against prediction bounds. Any data point outside the bounds is flagged.

In [ ]:
await run_agent("Detect any anomalies in the daily sales totals.")

### 5b. "Let me check the receipt" — Visual Audit (ObjectRef)

The agent looks up the flagged receipt by ID and returns a Cloud Storage URI. In production, this could link to scanned receipts, invoice PDFs, or product images — bridging structured data with unstructured files.

In [ ]:
await run_agent("Show me the receipt image link for the sale with receipt_id 'R_ANOMALY'.")

### 5c. "What happens next?" — Forecast Sales

Now that we've identified and investigated the anomaly, the agent forecasts the next 7 days using `AI.FORECAST` to help plan ahead.

In [ ]:
await run_agent("Forecast the total daily sales for the next 7 days.")

### 6. Key Takeaways

| Feature | What It Does | Partner Value |
|---------|-------------|---------------|
| **`AI.FORECAST`** | Zero-shot time-series forecasting via TimesFM | No `CREATE MODEL` step — any analyst can forecast without ML expertise |
| **`AI.DETECT_ANOMALIES`** | Flags outliers against a historical baseline | Real-time anomaly detection without training pipelines |
| **`ObjectRef`** | References GCS files (images, PDFs) from SQL | Bridges structured and unstructured data — the "missing link" for retail |
| **Job Labels** | Tags agent-initiated queries for cost tracking | Makes AI-driven costs visible in billing (`ca-bq-job: true`) |
| **ADK Runner** | Manages sessions and streams agent responses | Standard pattern across all March 2026 demos |

#### TimesFM vs. ARIMA_PLUS — When to Use Which

| | TimesFM (`AI.*` functions) | ARIMA_PLUS (`ML.*` functions) |
|---|---|---|
| **Setup** | Zero-shot — no training step | Requires `CREATE MODEL` |
| **Best for** | Speed, complex patterns, high-frequency data | Small datasets (<128 points), strict explainability |
| **Data requirement** | 128+ historical points | As few as 3 points |
| **Use case** | Modern apps, real-time dashboards | Compliance-heavy, statistical reporting |

> **Production tip:** TimesFM requires **128+ historical data points** for reliable results. With fewer points, `AI.DETECT_ANOMALIES` silently returns empty results.

### 7. Cleanup (Optional)

Remove the demo dataset and GCS bucket to avoid ongoing storage charges.

In [ ]:
# Uncomment and run to delete demo resources
# from google.cloud import bigquery, storage
#
# bq_client = bigquery.Client(project=project_id)
# bq_client.delete_dataset(f"{project_id}.march_demo", delete_contents=True, not_found_ok=True)
# print(f"Deleted dataset '{project_id}.march_demo'")
#
# storage_client = storage.Client(project=project_id)
# bucket = storage_client.get_bucket(f"{project_id}-receipts")
# bucket.delete(force=True)
# print(f"Deleted bucket '{project_id}-receipts'")